# Pairs Trading with Webull Bars

    Notebook นี้ replicate แนวทาง lecture/research notebook ของ QuantopianDoc ให้เข้ากับ Webull API โดยใช้ข้อมูล historical bars เป็น input หลัก.

    QuantopianDoc topics used as inspiration:
    - `Introduction_to_Pairs_Trading`
- `Integration_Cointegration_and_Stationarity`

    Webull/OpenAPI sources:
    - https://developer.webull.com/apis/docs/market-data-api/getting-started
- https://developer.webull.com/apis/docs/reference/broker-market-data-api/bars-using-get/
- https://github.com/nutdnuy/quantopiandoc/tree/main

    Concept:
    - สร้าง spread ระหว่างสองสินทรัพย์จากราคาปิด แล้วทดสอบ z-score signal เบื้องต้น
    - เริ่มจาก offline sample เพื่อ run ได้ทันที
    - เปิด live mode ได้เมื่อมี Webull credential และสิทธิ์ Market Data
    - ทุก output ถูกเขียนลง `outputs/quantopian-style/<slug>/`

    Safety:
    - ห้ามฝัง App Key, App Secret, token, account id จริงใน notebook
    - ใช้ Webull Market Data แบบ read-only เท่านั้น
    - ไม่ place, preview, replace, cancel order ใน notebook ชุดนี้
    - ผลวิจัยย้อนหลังไม่ใช่ forecast หรือคำสั่งซื้อขาย


## Setup and Webull-style Data Loader

### โค้ดช่องถัดไปทำอะไร

- import pandas, numpy และ Plotly สำหรับ research notebook
- สร้าง offline Webull-style OHLCV bars ที่ deterministic เพื่อให้ run ได้ทันที
- เตรียม optional live mode ผ่าน `WEBULL_QUANTOPIAN_LIVE=1`
- save output ลงโฟลเดอร์ประจำ notebook โดยไม่แตะ order/trading API


In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

NOTEBOOK_SLUG = "05-pairs-trading"
LIVE_MODE = os.getenv("WEBULL_QUANTOPIAN_LIVE", "0") == "1"
OUTPUT_ROOT = Path(os.getenv("WEBULL_QUANTOPIAN_OUTPUT_DIR", "outputs/quantopian-style"))
OUTPUT_DIR = OUTPUT_ROOT / NOTEBOOK_SLUG
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

THEME = {
    "background": "#121212",
    "surface": "#1D1D1D",
    "primary": "#69F0AE",
    "profit": "#00E676",
    "loss": "#FF5252",
    "benchmark": "#03DAC6",
    "grid": "rgba(255,255,255,0.08)",
    "text": "rgba(255,255,255,0.87)",
}


def build_sample_prices(symbols=("AAPL", "MSFT", "SPY", "TSLA"), periods=252) -> pd.DataFrame:
    dates = pd.bdate_range("2025-01-02", periods=periods)
    rows = []
    market = np.linspace(0, 1, periods)
    for index, symbol in enumerate(symbols):
        trend = 0.0002 + 0.00005 * index
        cycle = 0.006 * np.sin(np.linspace(0, 10 + index, periods))
        shocks = 0.004 * np.cos(np.linspace(0, 16 + index * 2, periods))
        beta_component = (0.8 + index * 0.15) * 0.002 * np.sin(market * 18)
        returns = trend + cycle + shocks + beta_component
        close = 100 * (1 + pd.Series(returns, index=dates)).cumprod()
        open_price = close.shift(1).fillna(close.iloc[0] * 0.997)
        high = pd.concat([open_price, close], axis=1).max(axis=1) * (1.003 + index * 0.0004)
        low = pd.concat([open_price, close], axis=1).min(axis=1) * (0.997 - index * 0.0003)
        volume = (
            1_000_000
            + index * 170_000
            + (np.sin(np.linspace(0, 20, periods)) + 1) * 120_000
        ).round()
        rows.append(
            pd.DataFrame(
                {
                    "symbol": symbol,
                    "date": dates,
                    "open": open_price.to_numpy(),
                    "high": high.to_numpy(),
                    "low": low.to_numpy(),
                    "close": close.to_numpy(),
                    "volume": volume,
                }
            )
        )
    return pd.concat(rows, ignore_index=True)


def load_webull_or_sample_prices(symbols=("AAPL", "MSFT", "SPY", "TSLA")) -> pd.DataFrame:
    if not LIVE_MODE:
        data = build_sample_prices(symbols=symbols)
        data.to_csv(OUTPUT_DIR / "offline_webull_style_prices.csv", index=False)
        return data

    from webull_lab.clients import build_data_client
    from webull_lab.config import load_settings
    from webull_lab.market_data import get_stock_bars

    settings = load_settings()
    data_client = build_data_client(settings)
    frames = []
    for symbol in symbols:
        payload = get_stock_bars(data_client, symbol, "D")
        frame = pd.DataFrame(payload).rename(columns={"time": "date"})
        frame["symbol"] = symbol
        frames.append(frame[["symbol", "date", "open", "high", "low", "close", "volume"]])
    data = pd.concat(frames, ignore_index=True)
    data["date"] = pd.to_datetime(data["date"], errors="coerce")
    for column in ["open", "high", "low", "close", "volume"]:
        data[column] = pd.to_numeric(data[column], errors="coerce")
    data = data.dropna(subset=["date", "open", "high", "low", "close"])
    data = data.sort_values(["symbol", "date"])
    data.to_csv(OUTPUT_DIR / "live_webull_prices.csv", index=False)
    return data


def close_matrix(price_data: pd.DataFrame) -> pd.DataFrame:
    return price_data.pivot(index="date", columns="symbol", values="close").sort_index()


def daily_returns(close: pd.DataFrame) -> pd.DataFrame:
    return close.pct_change().dropna()


def save_json(name: str, payload: dict) -> Path:
    path = OUTPUT_DIR / f"{name}.json"
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return path


def save_plot(fig: go.Figure, name: str) -> Path:
    fig.update_layout(
        template="plotly_dark",
        paper_bgcolor=THEME["background"],
        plot_bgcolor=THEME["background"],
        font={"color": THEME["text"]},
        xaxis={"gridcolor": THEME["grid"]},
        yaxis={"gridcolor": THEME["grid"]},
    )
    path = OUTPUT_DIR / f"{name}.html"
    fig.write_html(path, include_plotlyjs="cdn")
    return path


prices = load_webull_or_sample_prices()
close = close_matrix(prices)
returns = daily_returns(close)
print(
    {
        "live_mode": LIVE_MODE,
        "rows": len(prices),
        "symbols": sorted(prices["symbol"].unique()),
        "output_dir": str(OUTPUT_DIR),
    }
)


## Data Quality Snapshot

### โค้ดช่องถัดไปทำอะไร

- ตรวจช่วงวันที่และรายชื่อ symbol ที่โหลดมา
- คำนวณ mean return และ daily volatility เบื้องต้น
- save summary เป็น JSON เพื่อใช้เป็น artifact ตรวจซ้ำได้


In [ ]:
summary = {
    "sample_start": str(close.index.min().date()),
    "sample_end": str(close.index.max().date()),
    "symbols": list(close.columns),
    "mean_daily_return": returns.mean().round(6).to_dict(),
    "daily_volatility": returns.std().round(6).to_dict(),
}
save_json("data_summary", summary)
pd.DataFrame(summary["mean_daily_return"], index=["mean_daily_return"]).T


## Research Analysis

### โค้ดช่องถัดไปทำอะไร

- ทำ analysis เฉพาะหัวข้อของ notebook นี้
- export ตาราง/JSON/HTML chart ลง output folder
- แสดงผลลัพธ์สุดท้ายเป็น DataFrame หรือ summary ที่อ่านง่าย


In [ ]:
pair = ("AAPL", "MSFT")
log_prices = np.log(close[list(pair)])
x = log_prices[pair[1]].to_numpy()
y = log_prices[pair[0]].to_numpy()
X = np.column_stack([np.ones(len(x)), x])
intercept, hedge_ratio = np.linalg.lstsq(X, y, rcond=None)[0]
spread = pd.Series(y - (intercept + hedge_ratio * x), index=log_prices.index, name="spread")
zscore = (spread - spread.rolling(60).mean()) / spread.rolling(60).std()
signal = (-np.sign(zscore)).where(zscore.abs() > 1.0, 0).fillna(0)
spread_return = returns[pair[0]] - hedge_ratio * returns[pair[1]]
strategy_returns = signal.shift(1).fillna(0) * spread_return
equity = (1 + strategy_returns).cumprod()

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=zscore.index,
        y=zscore,
        name="Spread z-score",
        line={"color": THEME["primary"]},
    )
)
fig.add_hline(y=1, line_dash="dash", line_color=THEME["loss"])
fig.add_hline(y=-1, line_dash="dash", line_color=THEME["profit"])
plot_path = save_plot(fig, "pair_zscore")
save_json(
    "pairs_summary",
    {
        "pair": pair,
        "hedge_ratio": float(hedge_ratio),
        "final_equity": float(equity.iloc[-1]),
        "plot": str(plot_path),
    },
)
pd.DataFrame({"spread": spread, "zscore": zscore, "signal": signal, "equity": equity}).tail()


## Exercises

            1. ลอง pair อื่นแล้วเปรียบเทียบ hedge ratio
2. เพิ่ม stop rule เมื่อ z-score เกิน 3
3. แยก calibration window กับ trading window เพื่อลด lookahead

            ## Transfer to Real Webull Data

            1. ตั้งค่า `.env` ตาม README หลักของ repo
            2. ตั้ง `WEBULL_QUANTOPIAN_LIVE=1`
            3. ตรวจว่าบัญชี/แอปมี market data permission
            4. run notebook ใหม่ แล้วเทียบ offline sample กับ live Webull bars

            ## Common Mistakes

            - ใช้ parameter ที่ optimize จากทั้ง sample แล้วเรียกว่า out-of-sample
            - ลืม shift signal ก่อนคำนวณ return
            - มองข้าม fee, slippage, liquidity และ market data permission
            - สรุป historical backtest เป็น prediction
